In [2]:
import chromadb
from transformers import CLIPProcessor, CLIPModel
import torch
import numpy as np

/home/owen/Projects/local-rag/rag/lib64/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
client = chromadb.PersistentClient(path="../steam_avatars_db")

In [4]:
collection = client.get_or_create_collection(name="steam_avatars_collection")

In [5]:
collection.count()

1300

In [6]:
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 30552.04it/s]


In [7]:
def search(text, n_results=3):
    inputs = processor(text=[text], return_tensors="pt", padding=True)

    with torch.no_grad():
        outputs = model.get_text_features(**inputs)

    if hasattr(outputs, "pooler_output"):
        emb = outputs.pooler_output
    else:
        emb = outputs

    emb = emb[0] if emb.ndim > 1 else emb
    emb = emb / emb.norm(p=2)
    emb = emb.cpu().numpy().astype(np.float32)

    results = collection.query(query_embeddings=[emb.tolist()], n_results=n_results)
    return [result["url"] for result in results["metadatas"][0]] + results["distances"]

In [8]:
search("anime girl uwu")

['https://shared.fastly.steamstatic.com/community_assets/images/items/2543050/e884ca78dcef265c1dc33f18680ba9c5cec0b97a.gif',
 'https://shared.fastly.steamstatic.com/community_assets/images/items/1385730/9b963628af732a322561925cb43db0991ab098d1.gif',
 'https://shared.fastly.steamstatic.com/community_assets/images/items/1203420/34da95dd3d7b0f20508e2455553892696d4c7e03.gif',
 [0.7014083862304688, 0.7030007839202881, 0.7099677324295044]]

In [9]:
search("darkest dungeon")

['https://shared.fastly.steamstatic.com/community_assets/images/items/1850570/c87f2ceaeb16bad2d41f613d667edd22acf8d673.gif',
 'https://shared.fastly.steamstatic.com/community_assets/images/items/1940340/23759826491dbbaba18ddefb18dfea82c0b22cef.gif',
 'https://shared.fastly.steamstatic.com/community_assets/images/items/1940340/83e225ac0f2358da5f8fd7e39f4763b9d7316d24.gif',
 [0.6893454790115356, 0.6902289390563965, 0.6917121410369873]]

In [10]:
search("skeleton")

['https://shared.fastly.steamstatic.com/community_assets/images/items/220820/0f16dd1c402ebe796e4237fc0c047362444f924d.gif',
 'https://shared.fastly.steamstatic.com/community_assets/images/items/1845910/ebd97327779af08c3f731f85762b55a8c0299bd6.gif',
 'https://shared.fastly.steamstatic.com/community_assets/images/items/579720/14104ab4737413b2a495b2ab0cd1191933570e99.gif',
 [0.7073025703430176, 0.7095180749893188, 0.7137463092803955]]